### EXP 1

In [21]:
import gensim.downloader as api
model=api.load("glove-wiki-gigaword-50")

In [23]:
print(model.most_similar(positive=['walking','swim'],negative=['walk']))

[('swimming', 0.8071774244308472), ('swimmers', 0.7504438757896423), ('swims', 0.7454535365104675), ('surfing', 0.7409682273864746), ('biking', 0.7364492416381836), ('swam', 0.731138288974762), ('jumping', 0.7218191027641296), ('diving', 0.7187682390213013), ('kayaking', 0.7121444344520569), ('snowboarding', 0.7015390992164612)]


In [24]:
result=model.most_similar(positive=['king','woman'],negative=['man'])
print(f"'king'+'man'-'woman' is closest to:{result[0]}")

'king'+'man'-'woman' is closest to:('queen', 0.8523604273796082)


### EXP 2

In [1]:
import gensim.downloader as api
import numpy as np
from numpy.linalg import norm
print("Loading pre-trained word vectors...")
word_vectors=api.load("word2vec-google-news-300")
def explore_word_relationships(word1,word2,word3):
    try:
        vec1=word_vectors[word1]
        vec2=word_vectors[word2]
        vec3=word_vectors[word3]
        result_vector=vec1-vec2+vec3
        similar_words=word_vectors.similar_by_vector(result_vector,topn=10)
        filtered_words=[(word,similarity) for word,similarity in similar_words if word not in {word1,word2,word3}]
        print(f"\nWord Relationship :{word1}-{word2}+{word3}")
        print("Most similar words to the result(excluding input words):")
        for word,similarity in filtered_words[:5]:
            print(f"{word}:{similarity:4f}")
    except KeyError as e:
        print(f"Error:{e} not found in the vocabulary.")
explore_word_relationships("king","man","women")        

Loading pre-trained word vectors...

Word Relationship :king-man+women
Most similar words to the result(excluding input words):
queen:0.535494
kings:0.516231
queens:0.499536
kumaris:0.492385
princes:0.462333


### EXP 2

In [8]:
import gensim.downloader as api
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np

In [9]:
model=api.load("glove-wiki-gigaword-50")

In [10]:

words = ['queen','throne','prince','daughter','elizabeth','princess','kingdom','monarch','eldest','widow']
vector = np.array([model[word]for word in words])
fig = make_subplots(rows = 1,cols=2,subplot_titles = ('PCA Visualization','t-SNE Visualization'),horizontal_spacing = 0.1)
pca=PCA(n_components=2)
pca_vector = pca.fit_transform(vector)
tsne = TSNE(n_components=2,perplexity= 5, max_iter = 250 , random_state = 42)
tsne_vector = tsne.fit_transform(vector)
fig.add_trace(go.Scatter(x = pca_vector[:,0],y = pca_vector[:,1],mode = 'markers+text',text = words, textposition = "top center",marker = dict(size = 12,color = 'purple',symbol = 'diamond',line = dict(color='gold',width=2)),name = 'Words'),row=1,col=1)
fig.add_trace(go.Scatter(x = tsne_vector[:,0],y = tsne_vector[:,1],mode = 'markers+text',text = words, textposition = "top center",marker = dict(size = 12,color = 'purple',symbol = 'diamond',line = dict(color='gold',width=2)),name = 'Words'),row=1,col=2)
fig.update_layout(height = 600, width =1200,showlegend  = False , template = 'plotly_white',title_text = "Royal/Monarchy Word Embeddings Visualization",title_x = 0.5,font = dict(size=12),plot_bgcolor = 'rgba(240,240,255,0.95)',)
fig.update_xaxes(showgrid=True,gridwidth = 1,gridcolor = 'lightgrey')
fig.update_yaxes(showgrid=True,gridwidth = 1,gridcolor = 'lightgrey')
fig.show()              

TypeError: TSNE.__init__() got an unexpected keyword argument 'max_iter'

### EXP 3

In [1]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
sentences=["This is a legal document about contracts.",
           "The court will review the legal case.",
           "Medical professionals required specific training.",
           "This is a medical report about the patient."]
tokenized_sentences=[simple_preprocess(sentence)for sentence in sentences]
model=Word2Vec(sentences=tokenized_sentences,vector_size=50,window=5,min_count=1,workers=4,epochs=10)
print(model.wv.most_similar("legal"))

[('case', 0.19041602313518524), ('document', 0.045007091015577316), ('contracts', -0.010094511322677135), ('the', -0.014259982854127884), ('report', -0.02316688746213913), ('court', -0.043792981654405594), ('will', -0.044073157012462616), ('review', -0.09419942647218704), ('patient', -0.12276068329811096), ('required', -0.14990590512752533)]


### EXP 4

In [9]:
!pip install transformers

import gensim.downloader as api
from transformers import pipeline
import nltk
import string
from nltk.tokenize import word_tokenize

nltk.download('punkt')

print("Loading pre-trained word vectors...")
word_vectors = api.load("glove-wiki-gigaword-100")


def replace_keyword_in_prompt(prompt, keyword, word_vectors, topn=1):
    words = word_tokenize(prompt)
    enriched_words = []

    for word in words:
        cleaned_word = word.lower().strip(string.punctuation)

        if cleaned_word == keyword.lower():
            try:
                similar_words = word_vectors.most_similar(cleaned_word, topn=topn)

                if similar_words:
                    replacement_word = similar_words[0][0]
                    print(f"Replacing '{word}' -> '{replacement_word}'")
                    enriched_words.append(replacement_word)
                    continue

            except KeyError:
                print(f"Word '{cleaned_word}' not found in vocabulary. Using original word.")

        enriched_words.append(word)

    enriched_prompt = " ".join(enriched_words)
    print(f"\nEnriched prompt: {enriched_prompt}")

    return enriched_prompt


print("\nLoading GPT-2 model...")
generator = pipeline("text-generation", model="gpt2")


def generate_response(prompt, max_length=100):
    try:
        response = generator(prompt, max_length=max_length, num_return_sequences=1)
        return response[0]['generated_text']

    except Exception as e:
        print(f"Error generating response: {e}")
        return None
original_prompt = "who is king"
print(f"\nOriginal prompt: {original_prompt}")

key_term = "king"

enriched_prompt = replace_keyword_in_prompt(original_prompt, key_term, word_vectors)

print("\nGenerating response for the original prompt...")
original_response = generate_response(original_prompt)

print("\nOriginal prompt response:")
print(original_response)

print("\nGenerating response for the enriched prompt...")
enriched_response = generate_response(enriched_prompt)

print("\nEnriched prompt response:")
print(enriched_response)

print("\nComparison of responses")

print("\nOriginal prompt response length:", len(original_response))
print("Enriched prompt response length:", len(enriched_response))

print("\nOriginal prompt response detail:", original_response.count("."))
print("Enriched prompt response detail:", enriched_response.count("."))

[nltk_data] Downloading package punkt to /home/ai/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading pre-trained word vectors...

Loading GPT-2 model...


Downloading:   0%|          | 0.00/665 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/548M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

Downloading: 0.00B [00:00, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Original prompt: who is king
Replacing 'king' -> 'prince'

Enriched prompt: who is prince

Generating response for the original prompt...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Original prompt response:
who is king]

So, I shall ask a king, I shall ask what are the fruits of which you are an example, that I may bear with him, that all may remember me. And this I believe because my heart desires the fulfillment I cannot desire, because it is the fulfillment of my Father's faith.

4:4 And if I do not see my Father in any of the things which I have seen, yet do not believe, because I am lost in them

Generating response for the enriched prompt...

Enriched prompt response:
who is prince of the throne). They used their magic skill to break through the ice and make a pact with the demons to end all the trials and keep the evil from getting out; it wasn't quite as if they were the Chosen among them but when one of them dies they must leave, and they don't know who to go pick the last one away. I guess it's one hell of a series that seems to have a lot to do with "Maiden of the Shadowlands". If

Comparison of responses

Original prompt response length: 400
Enriched

### EXP 5

In [3]:
import gensim.downloader as api
import random
import nltk 
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/ai/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [7]:
word=api.load("glove-wiki-gigaword-100")
word_vectors = api.load("glove-wiki-gigaword-100") 
print("Word vectors loaded successfully!")
def get_similar_words(seed_word, top_n=5):
    try:
        similar_words = word_vectors.most_similar(seed_word, topn=top_n)
        return [word[0] for word in similar_words]
    except KeyError:
        print(f"'{seed_word}' not found in vocabulary. Try another word.")
        return []
def generate_sentence(seed_word, similar_words):
    sentence_templates = [
    f"The {seed_word} was surrounded by {similar_words[0]} and {similar_words[1]}.",
    f"People often associate {seed_word} with {similar_words[2]} and {similar_words[3]}.",
    f"In the land of {seed_word}, {similar_words[4]} was a common sight.",
    f"A story about {seed_word} would be incomplete without {similar_words[1]} and {similar_words[3]}.",
 ]
    return random.choice(sentence_templates)
def generate_paragraph(seed_word):
    similar_words = get_similar_words(seed_word, top_n=5)
    if not similar_words:
        return "Could not generate a paragraph. Try another seed word."
    paragraph = [generate_sentence(seed_word, similar_words) for _ in range(6)]
    return " ".join(paragraph)
seed_word = input("Enter a seed word: ")
paragraph = generate_paragraph(seed_word)
print("\nGenerated Paragraph:\n")
print(paragraph)       

Word vectors loaded successfully!
Enter a seed word: travel

Generated Paragraph:

The travel was surrounded by travelers and trips. People often associate travel with traveling and trip. The travel was surrounded by travelers and trips. People often associate travel with traveling and trip. In the land of travel, flights was a common sight. People often associate travel with traveling and trip.


### EXP 6

In [15]:
!pip install transformers
from transformers import pipeline
print("loading sentiment analysis model...")
sentiment_analyzer=pipeline("sentiment-analysis",model="distilbert-base-uncased-finetuned-sst-2-english",framework="pt")
def analyze_sentiment(text):
    result=sentiment_analyzer(text)[0]
    label=result['label']
    score=result['score']
    print(f"\nInput Text:{text}")
    print(f"sentiment:{label}(confidence:{score:.4f})\n")
    return result
customer_reviews=["The product is amazing! I love it so much",
                  "I am very disapppinted.The service was terrible.",
                  "It was an average experience,nothing special",
                  "Absolutely fantastic quality!highly recommended.",
                 "Not great,but not the work either."]
print("\n customer sentiment analysis results:")
for review in customer_reviews:
    analyze_sentiment(review)

loading sentiment analysis model...

 customer sentiment analysis results:

Input Text:The product is amazing! I love it so much
sentiment:POSITIVE(confidence:0.9999)


Input Text:I am very disapppinted.The service was terrible.
sentiment:NEGATIVE(confidence:0.9995)


Input Text:It was an average experience,nothing special
sentiment:NEGATIVE(confidence:0.9996)


Input Text:Absolutely fantastic quality!highly recommended.
sentiment:POSITIVE(confidence:0.9999)


Input Text:Not great,but not the work either.
sentiment:NEGATIVE(confidence:0.9942)



### EXP 7

In [2]:
!pip install transformers
from transformers import pipeline

loading summarization Model(BART)


Downloading: 0.00B [00:00, ?B/s]

Downloading:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Downloading: 0.00B [00:00, ?B/s]

Downloading: 0.00B [00:00, ?B/s]

Downloading: 0.00B [00:00, ?B/s]

In [3]:
print("loading summarization Model(BART)")
sum=pipeline("summarization",model="facebook/bart-large-cnn")

loading summarization Model(BART)


In [8]:
def sum_text(text,max_length=None,min_length=None):
    text=" ".join(text.split())
    if not max_length:
        max_length=min(len(text)//3,150)
    if not min_length:
        min_length=max(30,max_length//3)
    summary_1 = sum(text, max_length=150, min_length=30, do_sample=False)
    summary_2 = sum(text, max_length=150, min_length=30, do_sample=True, 
    temperature=0.9)
    summary_3 = sum(text, max_length=150, min_length=30, do_sample=False, 
    num_beams=5)
    summary_4 = sum(text, max_length=150, min_length=30, do_sample=True, top_k=50, 
    top_p=0.95)
    print("\nOriginal Text:")
    print(text)
    print("\nSummarized Text:")
    print("\nDefault:", summary_1[0]['summary_text'])
    print("\nHigh randomness:", summary_2[0]['summary_text'])
    print("\nConservative:", summary_3[0]['summary_text'])
    print("\nDiverse sampling:", summary_4[0]['summary_text'])
long_text="""Artificial intelligence is a rapidly evolving field of computer science focused on creating intelligent machines capable of mimicking human cognitive functions such as learning, problem-solving,and decision-making.In recent years AI has significantly impacted varoius industries,including healthcare,finance,education and entertainment.AI-powered applications such as chatbots,self-driving cars and recommendation systems have transformed the way we interact with technology Machine learning and deep learning,subsets of AI,enable systems to learn from the data and improve over time without explict programming.however, AI aslo poses ethical challenges,such as bias in decision-making and concerns over job displacement"""    
sum_text(long_text)    

Your max_length is set to 150, but you input_length is only 134. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)
Your max_length is set to 150, but you input_length is only 134. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)
Your max_length is set to 150, but you input_length is only 134. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)
Your max_length is set to 150, but you input_length is only 134. You might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)



Original Text:
Artificial intelligence is a rapidly evolving field of computer science focused on creating intelligent machines capable of mimicking human cognitive functions such aas learning, problem-solving,and decision-making.In recent years AI has significantly impacted varoius industries,including healthcare,finance,education and entertainment.AI-powered applications such as chatbots,self-driving cars and recommendation systems have transformed the way we interact with technology Machine learning and deep learning,subsets of AI,enable systems to learn from the data and improve over time without explict programming.however, AI aslo poses ethical challenges,such as bias in decision-making and concerns over job displacement

Summarized Text:

Default: Artificial intelligence is a rapidly evolving field of computer science. It is focused on creating intelligent machines capable of mimicking human cognitive functions. In recent years AI has significantly impacted varoius industries, 

### EXP 8 vP808ysuVydymX0RCIWJYXgnbSbdJ6EPD4SxPi1w

In [2]:
!pip uninstall -y numpy scipy langchain-text-splitters nltk
!pip install numpy scipy
!pip install -q langchain-core langchain-cohere cohere langchain-text-splitters nltk

# ============================================================
# CELL 1: Install only what's needed
# ============================================================
!pip install cohere

# ============================================================
# CELL 2: Imports
# ============================================================
import os
import getpass
import cohere

print("✅ All imports successful!")

# ============================================================
# CELL 3: Remove proxy settings
# ============================================================
for var in ['http_proxy', 'https_proxy', 'all_proxy',
            'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY']:
    os.environ.pop(var, None)

print("✅ Proxy settings cleared!")

# ============================================================
# CELL 4: Create a.txt with sample content
# ============================================================
sample_text = """
Artificial Intelligence is transforming the healthcare industry in remarkable ways.
Machine learning algorithms can now detect diseases like cancer at early stages with
high accuracy. AI-powered chatbots are helping patients get quick medical advice and
reducing the burden on doctors. Hospitals are using AI to optimize patient scheduling,
reduce wait times, and improve overall efficiency. However, there are concerns about
data privacy, algorithmic bias, and the need for human oversight in critical decisions.
Despite these challenges, the future of AI in healthcare looks very promising.
"""

with open("a.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("✅ a.txt created successfully!")

# ============================================================
# CELL 5: Load the text file
# ============================================================
file_path = "a.txt"

try:
    with open(file_path, "r", encoding="utf-8") as file:
        text_content = file.read()
    print(f"✅ File loaded! ({len(text_content)} characters)")
except FileNotFoundError:
    print(f"❌ '{file_path}' not found.")
    text_content = ""
except Exception as e:
    print(f"❌ Error: {e}")
    text_content = ""

# ============================================================
# CELL 6: Set up Cohere client
# ============================================================
COHERE_API_KEY = getpass.getpass("Enter your Cohere API Key: ")

client = cohere.ClientV2(api_key=COHERE_API_KEY)

print("✅ Cohere client initialized!")

# ============================================================
# CELL 7: Run analysis and print results
# ============================================================
if text_content:
    print("⏳ Analyzing document, please wait...\n")

    prompt = f"""
You are an AI assistant helping to summarize and analyze a text document.

Here is the document content:
{text_content}

Please provide the following:

Summary:
- Provide a concise summary of the document.

Key Takeaways:
- List 3 important points from the text.

Sentiment Analysis:
- Determine if the overall sentiment is Positive, Negative, or Neutral, and briefly explain why.
"""

    response = client.chat(
        model="command-a-03-2025",
        messages=[{"role": "user", "content": prompt}]
    )

    print("=" * 50)
    print("       DOCUMENT ANALYSIS RESULTS")
    print("=" * 50)
    print(response.message.content[0].text)
    print("=" * 50)

else:
    print("⚠️ No content to analyze. Please check that a.txt exists and is not empty.")
    


Found existing installation: numpy 1.24.4
Uninstalling numpy-1.24.4:
  Successfully uninstalled numpy-1.24.4
Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: nltk 3.9.4
Uninstalling nltk-3.9.4:
  Successfully uninstalled nltk-3.9.4
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.24.0 requires huggingface-hub<1.0,>=0.10.0, but you have huggingface-hub 1.9.2 which is incompatible.
transformers 4.24.0 requires to

### EXP 9 vP808ysuVydymX0RCIWJYXgnbSbdJ6EPD4SxPi1w

In [19]:
!pip install wikipedia-api

In [20]:
from pydantic import BaseModel
import wikipediaapi
from typing import List, Optional

In [21]:
class InstitutionDetails(BaseModel):
    founder: Optional[str]
    founded: Optional[str]
    branches: Optional[List[str]]
    number_of_employees: Optional[int]
    summary: Optional[str]

In [24]:
def fetch_institution_details(name: str) -> InstitutionDetails:
    wiki_wiki = wikipediaapi.Wikipedia(user_agent='your-user-agent', language='en') 
    page = wiki_wiki.page(name)
    if not page.exists():
        raise ValueError(f"'{name}' does not exist on Wikipedia.")
    info = {line.split(':')[0].strip(): line.split(':')[-1].strip() for line in page.text.split('\n') if ':' in line}
    return InstitutionDetails(founder=info.get('Founder', None), founded=info.get('Founded', None),
    branches=info.get('Branches', '').split(',') if 'Branches' in info else None,
    number_of_employees=int(info.get('Number of employees', '0').replace(',', '') or 0),
    summary=page.summary[:500])

In [26]:
def display_institution_details(details):
    print(
        f"Founder: {details.founder or 'N/A'}\n"
        f"Branches: {', '.join(details.branches) if details.branches else 'N/A'}\n"
        f"Employees: {details.number_of_employees or 'N/A'}\n"
        f"Summary: {details.summary or 'N/A'}"
    )

In [28]:
def main():
    institution_name=input("enter institution name: ").strip()
    if institution_name:
        try:
            details=fetch_institution_details(institution_name)
            display_institution_details(details)
        except ValueError as e:
            print(e)
    else:
        print("x please enter a valid institution name: ")
if __name__=="__main__":
    main()

enter institution name: harvard university
Founder: N/A
Branches: N/A
Employees: N/A
Summary: Harvard University is a private Ivy League research university in Cambridge, Massachusetts, United States. Founded in 1636, and named Harvard College in 1639 in honor of its first benefactor, Puritan clergyman John Harvard, it is the oldest institution of higher learning in the United States. Its influence, wealth, and rankings have made it one of the most prestigious universities in the world.
Harvard was founded and authorized by the Massachusetts General Court, the governing legislature of co


### EXP 10 vP808ysuVydymX0RCIWJYXgnbSbdJ6EPD4SxPi1w

In [34]:
!pip install langchain cohere langchain_community wikipedia-api pydantic ipywidgets
!pip install langchain-cohere
import getpass
from typing import Optional
from langchain_core.prompts import PromptTemplate
from langchain_cohere import ChatCohere
from pydantic import BaseModel
import wikipediaapi
from IPython.display import display
import ipywidgets as widgets

In [35]:
for var in ['http_proxy', 'https_proxy', 'all_proxy',
            'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY']:
    os.environ.pop(var, None)

print("✅ Proxy settings cleared!")

✅ Proxy settings cleared!


In [36]:
COHERE_API_KEY = getpass.getpass('Enter your Cohere API Key: ')
cohere_llm = ChatCohere(cohere_api_key=COHERE_API_KEY, model="command")
def fetch_ipc_summary():
    user_agent = "IPCChatbot/1.0 (contact: myemail@example.com)"
    wiki_wiki = wikipediaapi.Wikipedia(user_agent=user_agent, language='en')
    page = wiki_wiki.page("Indian Penal Code")
    if not page.exists():
        raise ValueError("The Indian Penal Code page does not exist on Wikipedia.")
    return page.text[:5000]  
ipc_content = fetch_ipc_summary()

Enter your Cohere API Key: ········


In [37]:
class IPCResponse(BaseModel):
    section: Optional[str]
    explanation: Optional[str]

prompt_template = PromptTemplate(input_variables=["ipc_content", "question"],
    template="""
You are a legal assistant chatbot specialized in the Indian Penal Code (IPC).
Refer to the following IPC document content to answer the user's question.
{ipc_content}
User Question: {question}
Provide a detailed answer, mentioning the relevant section if applicable.
""")

In [29]:
def get_ipc_response(question: str) -> IPCResponse:
    formatted_prompt = prompt_template.format(ipc_content=ipc_content, question=question)
    response = cohere_llm.invoke(formatted_prompt)
   
    if "Section" in response:
        section_start = response.find("Section")
        section = response[section_start:].split(":")[0].strip()
        explanation = response.split(":", 1)[-1].strip()
    else:
        section = None
        explanation = response.strip()
    return IPCResponse(section=section, explanation=explanation)

In [39]:
def display_response(response: IPCResponse):
    print(f" Section: {response.section if response.section else 'N/A'}")
    print(f" Explanation:\n{response.explanation}")

In [33]:
def on_button_click(b):
    user_question = text_box.value
    try:
        response = get_ipc_response(user_question)
        display_response(response)
    except Exception as e:
        print(f" Error: {e}")
text_box = widgets.Text(value='', placeholder='Ask about the Indian Penal Code...',
    description='You:', disabled=False)
button = widgets.Button(description='Ask', button_style='info',
    tooltip='Click to ask a question about IPC', icon='gavel')
button.on_click(on_button_click)
display(text_box, button)

Text(value='', description='You:', placeholder='Ask about the Indian Penal Code...')

Button(button_style='info', description='Ask', icon='gavel', style=ButtonStyle(), tooltip='Click to ask a ques…

 Error: [Errno -3] Temporary failure in name resolution
 Error: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '686d64915c6714d7d2203713cf648f31', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '40', 'x-trial-endpoint-call-remaining': '39', 'date': 'Thu, 23 Apr 2026 06:29:54 GMT', 'x-envoy-upstream-service-time': '9', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 404, body: {'id': 'fbc6a11f-5038-4680-aa0f-37b0239e4289', 'message': "model 'command' was removed on September 15, 2025. See https://docs.cohere.com/docs/models#command for a list of models you 

In [5]:
!pip install -q cohere wikipedia-api pydantic ipywidgets

import re
from typing import Optional
from pydantic import BaseModel
import wikipediaapi
import cohere
from IPython.display import display
import ipywidgets as widgets

COHERE_API_KEY = ""

co = cohere.Client(api_key=COHERE_API_KEY)

def fetch_ipc_summary():
    wiki = wikipediaapi.Wikipedia(
        language='en',
        extract_format=wikipediaapi.ExtractFormat.WIKI,
        user_agent="IPCChatbot/1.0 (contact: your_email@example.com)"
    )
    page = wiki.page("Indian Penal Code")
    if not page.exists():
        raise ValueError("IPC page not found")
    return page.text[:5000]

ipc_content = fetch_ipc_summary()

class IPCResponse(BaseModel):
    section: Optional[str] = None
    explanation: Optional[str] = None

def build_prompt(content, question):
    return f"""
You are an expert legal assistant specializing in the Indian Penal Code.

Use only the provided IPC content to answer accurately.

IPC Content:
{content}

Question: {question}

Provide a precise answer and include the relevant IPC section if applicable.
"""

def get_ipc_response(question: str) -> IPCResponse:
    prompt = build_prompt(ipc_content, question)
    response = co.chat(
        model="command-a-03-2025",
        message=prompt
    )
    text = response.text.strip()
    match = re.search(r"(Section\s*\d+)", text)
    section = match.group(1) if match else None
    return IPCResponse(section=section, explanation=text)

output_area = widgets.Output()

def render(response: IPCResponse):
    with output_area:
        output_area.clear_output()
        print(f"Section: {response.section if response.section else 'N/A'}\n")
        print(response.explanation)

def handle_click(_):
    query = input_box.value.strip()
    if not query:
        with output_area:
            output_area.clear_output()
            print("Enter a valid question")
        return
    try:
        res = get_ipc_response(query)
        render(res)
    except Exception as e:
        with output_area:
            output_area.clear_output()
            print(f"Error: {e}")

input_box = widgets.Text(
    placeholder='Ask about IPC...',
    description='Query:',
    layout=widgets.Layout(width='500px')
)

button = widgets.Button(
    description='Ask',
    button_style='info',
    icon='gavel'
)

button.on_click(handle_click)

display(input_box, button, output_area)

ValueError: Unknown scheme for proxy URL URL('socks://192.168.100.2:3128/')

In [3]:
import cohere
import wikipediaapi

co = cohere.Client("YOUR_API_KEY")

wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='my-wiki-app/1.0 (contact: your_email@example.com)'
)
page = wiki.page(" what is section 370")
text = page.text[:500]

print(text)

ValueError: Unknown scheme for proxy URL URL('socks://192.168.100.2:3128/')